# SQL Intermedio: Agregaciones, Joins y Subconsultas

En este cuaderno exploraremos conceptos intermedios de SQL utilizando Python y la librería `sqlite3` junto con `pandas` para visualizar los resultados.

In [ ]:
import sqlite3
import pandas as pd

# Crear una base de datos en memoria para el ejemplo
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Crear tablas de ejemplo: empleados y departamentos
cursor.execute('''
CREATE TABLE departamentos (
    id INTEGER PRIMARY KEY,
    nombre TEXT
);
''')

cursor.execute('''
CREATE TABLE empleados (
    id INTEGER PRIMARY KEY,
    nombre TEXT,
    departamento_id INTEGER,
    salario REAL,
    FOREIGN KEY (departamento_id) REFERENCES departamentos (id)
);
''')

# Insertar datos
cursor.executemany('INSERT INTO departamentos VALUES (?, ?)', [
    (1, 'Ventas'),
    (2, 'IT'),
    (3, 'Recursos Humanos'),
    (4, 'Marketing')
])

cursor.executemany('INSERT INTO empleados VALUES (?, ?, ?, ?)', [
    (1, 'Ana', 1, 3000),
    (2, 'Luis', 1, 3500),
    (3, 'Carlos', 2, 5000),
    (4, 'Sofia', 2, 6000),
    (5, 'Marta', 3, 2500),
    (6, 'Pedro', None, 2000)
])

conn.commit()

## 09. Funciones de Agregación (Aggregate Functions)
Las funciones de agregación realizan un cálculo sobre un conjunto de valores y devuelven un solo valor (COUNT, SUM, AVG, MAX, MIN).

In [ ]:
query = '''
SELECT 
    COUNT(*) AS total_empleados,
    SUM(salario) AS salario_total,
    AVG(salario) AS salario_promedio,
    MAX(salario) AS salario_maximo,
    MIN(salario) AS salario_minimo
FROM empleados;
'''
pd.read_sql_query(query, conn)

## 10. GROUP BY
Permite agrupar filas que comparten un mismo valor en columnas específicas para aplicar funciones de agregación a cada grupo.

In [ ]:
query = '''
SELECT 
    departamento_id,
    COUNT(*) AS num_empleados,
    AVG(salario) AS salario_promedio
FROM empleados
GROUP BY departamento_id;
'''
pd.read_sql_query(query, conn)

## 11. HAVING
Se utiliza para filtrar los resultados agrupados (similar a WHERE, pero operando sobre funciones de agregación).

In [ ]:
query = '''
SELECT 
    departamento_id,
    AVG(salario) AS salario_promedio
FROM empleados
GROUP BY departamento_id
HAVING AVG(salario) > 3000;
'''
pd.read_sql_query(query, conn)

## 12. INNER JOIN
Devuelve registros que tienen valores coincidentes en ambas tablas.

In [ ]:
query = '''
SELECT 
    e.nombre AS empleado,
    d.nombre AS departamento
FROM empleados e
INNER JOIN departamentos d 
    ON e.departamento_id = d.id;
'''
pd.read_sql_query(query, conn)

## 13. LEFT JOIN
Devuelve todos los registros de la tabla izquierda, y los registros coincidentes de la tabla derecha.

In [ ]:
query = '''
SELECT 
    e.nombre AS empleado,
    d.nombre AS departamento
FROM empleados e
LEFT JOIN departamentos d 
    ON e.departamento_id = d.id;
'''
pd.read_sql_query(query, conn)

## 14. RIGHT JOIN y FULL JOIN
Nota: SQLite no soporta directamente RIGHT JOIN ni FULL OUTER JOIN, pero se pueden emular con UNION o invirtiendo las tablas en un LEFT JOIN.

In [ ]:
# Emulando RIGHT JOIN invirtiendo la lógica con LEFT JOIN
query = '''
SELECT 
    e.nombre AS empleado,
    d.nombre AS departamento
FROM departamentos d
LEFT JOIN empleados e 
    ON e.departamento_id = d.id;
'''
pd.read_sql_query(query, conn)

## 15. SELF JOIN
Es una unión regular donde una tabla se une a sí misma.

In [ ]:
# Ejemplo teórico creando una relación de compañeros en el mismo departamento
query = '''
SELECT 
    e1.nombre AS empleado1,
    e2.nombre AS empleado2,
    e1.departamento_id
FROM empleados e1
JOIN empleados e2 
    ON e1.departamento_id = e2.departamento_id 
    AND e1.id < e2.id;
'''
pd.read_sql_query(query, conn)

## 16. Subconsultas (Subqueries)
Una consulta dentro de otra consulta.

In [ ]:
query = '''
SELECT nombre, salario
FROM empleados
WHERE salario > (
    SELECT AVG(salario) FROM empleados
);
'''
pd.read_sql_query(query, conn)

## 17. CTEs (Common Table Expressions)
Las CTEs proporcionan un conjunto de resultados temporal con nombre que se puede referenciar dentro de una sentencia SELECT, INSERT, UPDATE o DELETE.

In [ ]:
query = '''
WITH PromedioSalarios AS (
    SELECT departamento_id, AVG(salario) as promedio
    FROM empleados
    GROUP BY departamento_id
)
SELECT 
    e.nombre,
    e.salario,
    p.promedio
FROM empleados e
JOIN PromedioSalarios p 
    ON e.departamento_id = p.departamento_id
WHERE e.salario > p.promedio;
'''
pd.read_sql_query(query, conn)

In [ ]:
# Cerrar la conexión
conn.close()